In [ ]:
import importlib.metadata as md

for paket in ["imagehash", "pillow", "tensorflow", "numpy", "matplotlib"]:
    print(f"{paket}=={md.version(paket)}")

# Cek Duplikat Dataset

Memeriksa foto yang sama atau sangat mirip di dataset (train dan val) memakai *perceptual hash*. Duplikat, terutama yang tersebar di train dan val, membuat akurasi validasi terlihat lebih bagus dari kenyataan.

Jalankan ulang setiap kali foto baru ditambahkan, dan **sebelum training**.

In [ ]:
import os
from PIL import Image
import imagehash

DATASET_DIR = "../dataset"
KELAS = ["rendah", "sedang", "tinggi"]
EKSTENSI = (".jpg", ".jpeg", ".png")

hash_gambar = {}
for split in ["train", "val"]:
    for kelas in KELAS:
        folder = os.path.join(DATASET_DIR, split, kelas)
        if not os.path.isdir(folder):
            continue
        for nama in os.listdir(folder):
            if nama.lower().endswith(EKSTENSI):
                path = os.path.join(folder, nama)
                hash_gambar[path] = imagehash.phash(Image.open(path))

print("Jumlah gambar yang dihitung:", len(hash_gambar))

In [ ]:
BATAS = 11

daftar = list(hash_gambar.items())
pasangan = []
for i in range(len(daftar)):
    for j in range(i + 1, len(daftar)):
        jarak = daftar[i][1] - daftar[j][1]
        if jarak <= BATAS:
            pasangan.append((jarak, daftar[i][0], daftar[j][0]))

pasangan.sort()
print("Pasangan mirip:", len(pasangan))
for jarak, a, b in pasangan:
    print(jarak, "|", a, "|", b)

In [ ]:
import matplotlib.pyplot as plt

for jarak, a, b in pasangan[:6]:
    plt.figure(figsize=(8, 4))
    for k, p in enumerate([a, b]):
        plt.subplot(1, 2, k + 1)
        plt.imshow(Image.open(p))
        plt.title(os.path.relpath(p, DATASET_DIR), fontsize=8)
        plt.axis("off")
    plt.suptitle(f"jarak = {jarak}")
    plt.show()